# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing **Lane 2: Refresh / Content Opportunity Scoring** with **individual pages** (content items) as the unit of analysis.

Our goal is to build a prioritized queue of pages that are underperforming or declining but have sufficient underlying search demand to justify a manual update. Focusing on individual pages makes the recommendations highly actionable for content editors, who can revise, expand, or prune specific URLs. This lane directly impacts editorial ROI by pointing limited human resources to where they can prevent the most traffic decay or capture the most striking-distance search volume.

In [1]:
# Check the distribution of clients and pages in the starter dataset
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Total Pages: {len(df)}")
print(f"Total Clients: {df['client_id'].nunique()}")
print(f"Average Pages per Client: {len(df) / df['client_id'].nunique():.1f}")


Total Pages: 30000
Total Clients: 32
Average Pages per Client: 937.5


## 2. The question: decision, action, cost of a wrong call

- **What decision does this improve?** It improves the decision of how a content operations team or editor allocates their limited time and budget: specifically, "Which pages should we refresh first to prevent traffic loss or recover search visibility?"
- **Who acts on the output, and what do they do?** The content editor or SEO manager acts on the output. They will take specific actions based on the page's reason codes, such as updating out-of-date facts, expanding thin content, optimizing meta titles/descriptions for low CTR, or merging cannibalizing pages.
- **What does a wrong recommendation cost?** A false positive (recommending a page that doesn't need a refresh or cannot be improved) costs wasted editorial hours and budget (typically $150-$500 per updated article). A false negative (missing a critical page that is in severe decline) costs lost traffic, lower conversions, and a long-term decline in search ranking that is harder to recover from.
- **Why does data or ML help?** With thousands of pages per client, editors cannot manually audit every URL daily. Standard rule-based filters (e.g., "refresh anything older than 180 days") are too blunt and result in wasted effort on low-impact pages. ML helps by modeling complex, interacting signals (impressions, CTR, position, engagement, and staleness) to score and prioritize opportunities based on real traffic risk and demand.

In [2]:
# Show the cost implications of naive rule-based decisions vs demand-aware targeting
total_stale = (df['days_since_last_update'] >= 180).sum()
stale_low_demand = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] < 100)).sum()
stale_high_demand = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 100)).sum()

print(f"Total stale pages (last updated >= 180 days ago): {total_stale}")
print(f"Stale pages with LOW demand (< 100 impressions/90d) - risk of wasted refresh budget: {stale_low_demand}")
print(f"Stale pages with HIGH demand (>= 100 impressions/90d) - priority refresh targets: {stale_high_demand}")


Total stale pages (last updated >= 180 days ago): 174
Stale pages with LOW demand (< 100 impressions/90d) - risk of wasted refresh budget: 139
Stale pages with HIGH demand (>= 100 impressions/90d) - priority refresh targets: 35


## 3. Quick look at the data (2-3 real numbers)

To support the choice of Lane 2, I ran a quick analysis of the starter dataset `content_refresh_anonymized.csv`:
1. **Decline rate**: Out of 30,000 pages, **16,262 (54.2%)** are currently in decline (`trend_direction == 'down'`).
2. **Staleness and decline**: There are **9,345 pages (31.2%)** that have not been updated for 90 days or more. Among these, **5,686 pages (60.8% of the stale pages)** are in decline, highlighting a strong association between content age/staleness and traffic loss.
3. **High-Impact Opportunity**: **16,726 pages (55.8%)** have significant search demand (at least 500 impressions over 90 days), of which **9,961** are declining. This confirms a massive pool of high-visibility pages that are losing traction and would yield the highest return on investment if refreshed.

In [3]:
# Compute the real numbers supporting the lane choice
total_pages = len(df)
declining_pages = (df['trend_direction'] == 'down').sum()
stale_pages_90d = (df['days_since_last_update'] >= 90).sum()
stale_and_declining = ((df['trend_direction'] == 'down') & (df['days_since_last_update'] >= 90)).sum()
high_demand_pages = (df['impressions_90d'] >= 500).sum()
high_demand_declining = ((df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)).sum()

print(f"1. Overall Decline Rate: {declining_pages / total_pages * 100:.1f}% ({declining_pages}/{total_pages} pages)")
print(f"2. Staleness (>= 90 days) count: {stale_pages_90d} ({stale_pages_90d / total_pages * 100:.1f}%)")
print(f"   Stale pages currently declining: {stale_and_declining} ({stale_and_declining / stale_pages_90d * 100:.1f}% of stale pages)")
print(f"3. High-demand pages (>= 500 impressions): {high_demand_pages} ({high_demand_pages / total_pages * 100:.1f}%)")
print(f"   High-demand pages declining: {high_demand_declining} ({high_demand_declining / high_demand_pages * 100:.1f}% of high-demand pages)")


1. Overall Decline Rate: 54.2% (16262/30000 pages)
2. Staleness (>= 90 days) count: 9345 (31.1%)
   Stale pages currently declining: 5686 (60.8% of stale pages)
3. High-demand pages (>= 500 impressions): 16726 (55.8%)
   High-demand pages declining: 9961 (59.6% of high-demand pages)


## 4. Careful words: what I can and can't claim

- **What I CAN claim**: 
  - I can claim **observational and directional** findings. For example, "Pages with characteristics X, Y, Z are statistically more likely to experience a decline in search traffic."
  - I can claim to build a **decision-support tool** that ranks pages by prioritization score to optimize editor workflow.
- **What I CANNOT claim**:
  - I cannot claim **causal proof** (e.g., "Updating this page will cause its traffic to recover by X%"). To prove causality, we would need controlled A/B testing or randomized experiments, which this historical snapshot does not provide.
  - I cannot claim to **predict the Google Search Algorithm** or reverse-engineer search engine ranking functions. We are modeling the empirical relationship between our features and observed page outcomes, not Google's internal logic.

In [4]:
# Check that the pseudonyms are not treated as features and examine basic correlations
# Note: impressions_last_30d and other trend-input columns are leakage risks for prediction models.
numeric_cols = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] 
                and col not in ['trend_pct', 'is_declining_label'] 
                and not col.endswith('_id') and not col.endswith('_hash_id')]

correlations = df[numeric_cols].corrwith(df['trend_direction'].apply(lambda x: 1 if x == 'down' else 0))
print("Top 5 numeric correlations with decline status (absolute values):")
print(correlations.abs().sort_values(ascending=False).head(5))


Top 5 numeric correlations with decline status (absolute values):
days_with_impressions    0.190055
content_age_days         0.163882
age_tier_order           0.156142
impressions_last_30d     0.093980
word_count               0.090157
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.